In [152]:
import pytreenet as ptn
from copy import deepcopy
import numpy as np
import cmath
#np.random.seed(57643)

In [153]:
from qutip import coherent
def product_state(ttn, bond_dim=2 , physical_dim= 2):
    product_state = deepcopy(ttn)
    #A = np.array([1/np.sqrt(2),1/np.sqrt(2)])
    A = np.array([3,4j])
    alpha = 1
    A = np.array(coherent(physical_dim , alpha).full())
    for node_id in product_state.nodes.keys():
        n = product_state.tensors[node_id].ndim - 1
        tensor = A.reshape((1,) * n + (physical_dim,))
        T = np.pad(tensor, n*((0, 0),) + ((0, 0),))
        product_state.tensors[node_id] = T
        product_state.nodes[node_id].link_tensor(T)  
    return product_state

# Initialize vectorized_pho

In [154]:
# local physical dimension
d = 2

shapes = {
    (0, 0): (3, 5, 6, d),
    (0, 1): (3, 7, d),
    (0, 2): (7, 8, d),
    (1, 0): (5, 5, d),
    (1, 1): (9, d),
    (1, 2): (8, d),
    (2, 0): (5, 6, d),
    (2, 1): (6, 9, 3, d),
    (2, 2): (3, d)
}


sites = {
    (i, j): ptn.random_tensor_node(shapes[(i, j)], identifier=f"Site({i},{j})") for i in range(3) for j in range(3)
}

vectorized_pho = ptn.TreeTensorNetworkState()

vectorized_pho.add_root(sites[(0, 0)][0], sites[(0, 0)][1])

connections = [
    ((0, 0), (0, 1), 0, 0),
    ((0, 1), (0, 2), 1, 0),
    ((0, 2), (1, 2), 1, 0),
    ((0, 0), (1, 0), 1, 0),
    ((1, 0), (2, 0), 1, 0),
    ((2, 0), (2, 1), 1, 0),
    ((2, 1), (1, 1), 1, 0),
    ((2, 1), (2, 2), 2, 0)]


for (parent, child, parent_leg, child_leg) in connections:
    parent_id = f"Site({parent[0]},{parent[1]})"
    child_id = f"Site({child[0]},{child[1]})"
    vectorized_pho.add_child_to_parent(sites[child][0], sites[child][1], child_leg, parent_id, parent_leg)

vectorized_pho = product_state(vectorized_pho , bond_dim=4, physical_dim = d)

nodes = {
    (i, j): (ptn.Node(tensor=vectorized_pho.tensors[f"Site({i},{j})"] , identifier=f"Node({i},{j})"), vectorized_pho.tensors[f"Site({i},{j})"]) for i in range(3) for j in range(3)
}

vectorized_pho.add_child_to_parent(nodes[(0,0)][0], nodes[(0,0)][1], 2, "Site(0,0)", 2)

connections = [
    ((0, 0), (0, 1), 1, 0),
    ((0, 1), (0, 2), 1, 0),
    ((0, 2), (1, 2), 1, 0),
    ((0, 0), (1, 0), 2, 0),
    ((1, 0), (2, 0), 1, 0),
    ((2, 0), (2, 1), 1, 0),
    ((2, 1), (1, 1), 1, 0),
    ((2, 1), (2, 2), 2, 0),]

for (parent, child, parent_leg, child_leg) in connections:
    parent_id = f"Node({parent[0]},{parent[1]})"
    vectorized_pho.add_child_to_parent(nodes[child][0], nodes[child][1], child_leg, parent_id, parent_leg)

In [155]:
def get_neighbors_Site(x, y, Lx, Ly):
  neighbors = []
  
  # Right neighbor
  if x < Lx - 1:
      neighbors.append(f"Site({x+1},{y})")
  
  # Up neighbor
  if y < Ly - 1:
      neighbors.append(f"Site({x},{y+1})")
  
  return neighbors

def get_neighbors_Node(x, y, Lx, Ly):
  neighbors = []

  # Right neighbor
  if x < Lx - 1:
      neighbors.append(f"Node({x+1},{y})")
  
  # Up neighbor
  if y < Ly - 1:
      neighbors.append(f"Node({x},{y+1})")
  
  return neighbors

In [156]:
def Liouville(t, U, gamma, m, L, Lx, Ly, d):
    creation_op, annihilation_op, number_op = ptn.bosonic_operators(d)
    
    conversion_dict = {
        "b^dagger": creation_op,
        "b": annihilation_op,
        f"I{d}": np.eye(d)
    }
    
    conversion_dict.update({
        "it * b^dagger": t*1j * creation_op,
        "it * b": t*1j * annihilation_op,
        "-iU * n * (n - 1)": -U*1j * number_op @ (number_op - np.eye(d)),
        "im*n": m*1j*number_op
    })
    
    terms = []
    
    # Hopping terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Site({x},{y})"
            neighbors = get_neighbors_Site(x, y, Lx, Ly)            
            for neighbor in neighbors:
                terms.append(ptn.TensorProduct({current_site: "it * b^dagger", neighbor: "b"}))
                terms.append(ptn.TensorProduct({current_site: "it * b", neighbor: "b^dagger"}))
                

    
    # On-site interaction terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Site({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "-iU * n * (n - 1)"}))

    # Chemical potential terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Site({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "im*n"}))        
    
    H1 = ptn.Hamiltonian(terms, conversion_dict)
    
    conversion_dict = {
        "b^dagger.T": creation_op.T,
        "b.T": annihilation_op.T,
        f"I{d}": np.eye(d)
    }
    
    conversion_dict.update({
        "-it * b^dagger.T": -t*1j * creation_op.T,
        "-it * b.T": -t*1j * annihilation_op.T,
        "iU * n * (n - 1).T": (U*1j * number_op @ (number_op - np.eye(d))).T,
        "-im*n.T": -m*1j* number_op.T
    })
    
    terms = []
    
    # Hopping terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Node({x},{y})"
            neighbors = get_neighbors_Node(x, y, Lx, Ly)
            for neighbor in neighbors:
                terms.append(ptn.TensorProduct({current_site: "-it * b^dagger.T", neighbor: "b.T"}))
                terms.append(ptn.TensorProduct({current_site: "-it * b.T", neighbor: "b^dagger.T"}))

    # On-site interaction terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Node({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "iU * n * (n - 1).T"}))    

    # Chemical potential terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Node({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "-im*n.T"}))
            
    H2 = ptn.Hamiltonian(terms, conversion_dict)
    H1.__add__(H2)

        
    conversion_dict = {    
    "L": np.sqrt(gamma) * L,
    "L^dagger.T": np.sqrt(gamma) * L.conj(),
    "-1/2 (L^dagger @ L) " : -1/2 * gamma * L.conj().T @ L,
    "-1/2 (L^dagger @ L).T": -1/2 * gamma * (L.conj().T @ L).T}
    terms = []
    for x in range(Lx):
        for y in range(Ly):
            out_site = f"Node({x},{y})"
            in_site = f"Site({x},{y})"
            terms.append(ptn.TensorProduct({in_site: "L" , out_site: "L^dagger.T"}))
            terms.append(ptn.TensorProduct({in_site: "-1/2 (L^dagger @ L) "}))
            terms.append(ptn.TensorProduct({out_site: "-1/2 (L^dagger @ L).T"}))

    H3 = ptn.Hamiltonian(terms, conversion_dict)
    H1.__add__(H3)
    return H1

In [145]:
def Number_op_total(Lx, Ly, dim=2):
    creation_op, annihilation_op, number_op = ptn.bosonic_operators(dim)
    conversion_dict = {"n": number_op , f"I{dim}": np.eye(dim)}
    for dim in range(1, 200):
        conversion_dict[f"I{dim}"] = np.eye(dim)

    terms = []
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Vertex({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "n"}))

    return ptn.Hamiltonian(terms, conversion_dict) 

def BoseHubbard_ham(t, U, m, Lx, Ly, d):
    creation_op, annihilation_op, number_op = ptn.bosonic_operators(d)

    conversion_dict = {
        "b^dagger": creation_op,
        "b": annihilation_op,
        f"I{d}": np.eye(d)}
    
    conversion_dict.update({
        "-t * b^dagger": -t * creation_op,
        "-t * b": -t * annihilation_op,
        "U * n * (n - 1)": U * number_op @ (number_op - np.eye(d)),
        "-m*n": -m * number_op
    })

    for dim in range(2, 200):
        conversion_dict[f"I{dim}"] = np.eye(dim)    
    terms = []
    
    # Hopping terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Site({x},{y})"
            neighbors = get_neighbors_Site(x, y, Lx, Ly)
            for neighbor in neighbors:
                terms.append(ptn.TensorProduct({current_site: "-t * b^dagger", neighbor: "b"}))
                terms.append(ptn.TensorProduct({current_site: "-t * b", neighbor: "b^dagger"}))


    # On-site interaction terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Site({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "U * n * (n - 1)"}))
    
    # Chemical potential terms
    for x in range(Lx):
        for y in range(Ly):
            current_site = f"Site({x},{y})"
            terms.append(ptn.TensorProduct({current_site: "-m*n"}))

    return ptn.Hamiltonian(terms, conversion_dict)



# Define SecondOrderOneSiteTDVP
gamma = 0

In [146]:
t = 0.4
U = 0.8
m = 0.4
creation_op, annihilation_op, number_op = ptn.bosonic_operators(d)
L = annihilation_op
gamma = 1

ket , bra = ptn.devectorize_pho(vectorized_pho , connections)

# TTNO : Number operator
N = Number_op_total(3, 3, d)
N = N.pad_with_identities(ket, symbolic= True)
N = ptn.TTNO.from_hamiltonian(N, ket , ptn.TTNOFinder.TREE )

# TTNO : Bose-Hubbard Hamiltonian
H = BoseHubbard_ham(t, U, m, 3, 3, d)
H = H.pad_with_identities(vectorized_pho, symbolic= True)
H = ptn.TTNO.from_hamiltonian(H, vectorized_pho , ptn.TTNOFinder.TREE)

# TTNO : Identity operator
I_pho = ptn.TTNO.Identity(vectorized_pho)
I_ket = ptn.TTNO.Identity(ket)
# TTNO : Liouville operator 
H1 = Liouville(t, U, gamma, m ,L, 3, 3, d)
H1 = H1.pad_with_identities(vectorized_pho , symbolic= True)
L_fancy = ptn.TTNO.from_hamiltonian(H1, vectorized_pho ,  ptn.TTNOFinder.TREE)

connections = [ ((0, 0), (0, 1), 1, 0),
                ((0, 1), (0, 2), 1, 0),
                ((0, 2), (1, 2), 1, 0),
                ((0, 0), (1, 0), 2, 0),
                ((1, 0), (2, 0), 1, 0),
                ((2, 0), (2, 1), 1, 0),
                ((2, 1), (1, 1), 1, 0),
                ((2, 1), (2, 2), 2, 0)]

#vectorized_pho = ptn.normalize_ttn_Lindblad_X(vectorized_pho, "Site(0,0)" , connections)
tdvp_Lindblad = ptn.SecondOrderOneSiteTDVP(initial_state = vectorized_pho,
                                            hamiltonian = L_fancy,
                                            time_step_size = 0.001,
                                            final_time = 0.1,
                                            operators = N,
                                            connections = connections)

In [147]:
tdvp_Lindblad = ptn.FirstOrderOneSiteTDVP(initial_state = vectorized_pho,
                                            hamiltonian = L_fancy,
                                            time_step_size = 0.01,
                                            final_time = 1,
                                            operators = N,
                                            connections = connections)

In [148]:
stop

NameError: name 'stop' is not defined

In [ ]:
tdvp_Lindblad.run_Lindblad(evaluation_time=1)
times = tdvp_Lindblad.times

#  Initial state = np.array([3j,1j])


In [10]:
print(" METHOD 1 : tdvp.state is updated at each step with (|rho>>'s orthogonality_center)/<bra|ket> ")

print(" ########## initialize the TDVP ###########")
tdvp_Lindblad.state = ptn.normalize_ttn_Lindblad_X(tdvp_Lindblad.state, tdvp_Lindblad.update_path[0] , tdvp_Lindblad.connections)
tdvp_Lindblad._orthogonalize_init()
tdvp_Lindblad.partial_tree_cache = tdvp_Lindblad._init_partial_tree_cache()
tdvp_Lindblad.adjust_to_initial_structure()

print(" #########     Initial state     ##########") 
print(" <O>                           : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N))
print(" <bra|ket> , np.abs(<bra|ket>) : " , ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) , np.abs(ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) ) )
print(" <O> /<bra|ket>                : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N) / ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) )

for i in range(10):
    print(" |||||||||||||||||| START STEP : " , i , " ||||||||||||||||||||||")

    print(" ######### RUN ONE STEP #########")
    tdvp_Lindblad.run_one_time_step()
    tdvp_Lindblad.adjust_to_initial_structure()
    print(" <O>                           : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N))
    print(" <bra|ket> , np.abs(<bra|ket>) : " , ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) , np.abs(ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) ) )
    print(" <O> /<bra|ket>                : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N) / ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) )

    print(" ######### NORMALIZATION ##########")
    tdvp_Lindblad.state = ptn.normalize_ttn_Lindblad_X(tdvp_Lindblad.state, tdvp_Lindblad.update_path[0] , tdvp_Lindblad.connections)
    tdvp_Lindblad.adjust_to_initial_structure()
    print(" <O>                           : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N))
    print(" <bra|ket> , np.abs(<bra|ket>) : " , ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) , np.abs(ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) ) )
    print(" <O> /<bra|ket>                : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N) / ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) )


    print(" ######### orth. init and update treecache dict. with normalized state ##########") 
    tdvp_Lindblad._orthogonalize_init(force_new=True)
    tdvp_Lindblad.partial_tree_cache = ptn.PartialTreeCachDict()
    tdvp_Lindblad.partial_tree_cache = tdvp_Lindblad._init_partial_tree_cache()
    print(" <O>                           : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N))
    print(" <bra|ket> , np.abs(<bra|ket>) : " , ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) , np.abs(ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) ) )
    print(" <O> /<bra|ket>                : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N) / ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) )

    print(" ||||||||||||||||||| END STEP  : " , i , " |||||||||||||||||||||")



 METHOD 1 : tdvp.state is updated at each step with (|rho>>'s orthogonality_center)/<bra|ket> 
 ########## initialize the TDVP ###########
 #########     Initial state     ##########
 <O>                           :  (6.372660764462142+0j)
 <bra|ket> , np.abs(<bra|ket>) :  (1+0j) 1.0
 <O> /<bra|ket>                :  (6.372660764462142+0j)
 |||||||||||||||||| START STEP :  0  ||||||||||||||||||||||
 ######### RUN ONE STEP #########
 <O>                           :  (6.715785354982943+0.0021204073782844545j)
 <bra|ket> , np.abs(<bra|ket>) :  (1.046483768616799+3.4943691966713205e-05j) 1.0464837692002105
 <O> /<bra|ket>                :  (6.417476906665437+0.0018119316313531624j)
 ######### NORMALIZATION ##########
 <O>                           :  (6.417476771348212+0.002240510345311j)
 <bra|ket> , np.abs(<bra|ket>) :  (0.9999999977700119+6.678305574474079e-05j) 1.0
 <O> /<bra|ket>                :  (6.41747690666544+0.0018119316313531615j)
 ######### orth. init and update treecache dic

In [167]:
print(" METHOD 2 : Since normalization is METHOD 1, add a phase to state, here, I transfprmed the")
print("            state into two-site-canonial form centered at Site(0,0) and Node(0,0) and divided both")
print("            by np.sqrt(<bra|ket>) and np.sqrt(<bra|ket>).conj()")

print(" ########## initialize the TDVP ###########")
tdvp_Lindblad.state = ptn.normalize_ttn_Lindblad_XX(tdvp_Lindblad.state, "Site(0,0)" , "Node(0,0)", tdvp_Lindblad.connections )
tdvp_Lindblad._orthogonalize_init()
tdvp_Lindblad.partial_tree_cache = tdvp_Lindblad._init_partial_tree_cache()
tdvp_Lindblad.adjust_to_initial_structure()

print(" <O>                           : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N))
print(" <bra|ket> , np.abs(<bra|ket>) : " , ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) , np.abs(ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) ) )
print(" <O> /<bra|ket>                : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N) / ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) )

for i in range(10):
    print(" ||||||||||||||||||| START STEP : " , i , " ||||||||||||||||||||||")

    print(" ######### RUN ONE STEP #########")
    tdvp_Lindblad.run_one_time_step()
    tdvp_Lindblad.adjust_to_initial_structure()
    print(" <O>                           : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N))
    print(" <bra|ket> , np.abs(<bra|ket>) : " , ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) , np.abs(ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) ) )
    print(" <O> /<bra|ket>                : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N) / ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) )

    print(" ######### NORMALIZATION ##########")
    tdvp_Lindblad.state = ptn.normalize_ttn_Lindblad_XX(tdvp_Lindblad.state, "Site(0,0)" , "Node(0,0)" , tdvp_Lindblad.connections )
    print(" <O>                           : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N))
    print(" <bra|ket> , np.abs(<bra|ket>) : " , ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) , np.abs(ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) ) )
    print(" <O> /<bra|ket>                : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N) / ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) )

    print(" ######### orth. init and update treecache dict. with normalized state ##########") 
    tdvp_Lindblad._orthogonalize_init(force_new=True)
    tdvp_Lindblad.partial_tree_cache = ptn.PartialTreeCachDict()
    tdvp_Lindblad.partial_tree_cache = tdvp_Lindblad._init_partial_tree_cache()
    print(" <O>                           : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N))
    print(" <bra|ket> , np.abs(<bra|ket>) : " , ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) , np.abs(ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) ) )
    print(" <O> /<bra|ket>                : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N) / ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) )

    print(" ||||||||||||||||||| END STEP    : " , i , " |||||||||||||||||||||")




 METHOD 2 : Since normalization is METHOD 1, add a phase to state, here, I transfprmed the
            state into two-site-canonial form centered at Site(0,0) and Node(0,0) and divided both
            by np.sqrt(<bra|ket>) and np.sqrt(<bra|ket>).conj()
 ########## initialize the TDVP ###########
 <O>                           :  (5.759999999999998+0j)
 <bra|ket> , np.abs(<bra|ket>) :  (0.9999999999999998+0j) 0.9999999999999998
 <O> /<bra|ket>                :  (5.759999999999999+0j)
 ||||||||||||||||||| START STEP :  0  ||||||||||||||||||||||
 ######### RUN ONE STEP #########
 <O>                           :  (5.33123974116633+0.0038076193886074474j)
 <bra|ket> , np.abs(<bra|ket>) :  (0.9246283439923804-8.447829334327857e-06j) 0.9246283440309718
 <O> /<bra|ket>                :  (5.765819034828419+0.004170679028868356j)
 ######### NORMALIZATION ##########
 <O>                           :  (5.765819034828416+0.004170679028868353j)
 <bra|ket> , np.abs(<bra|ket>) :  (1-6.945670167485263e

In [177]:
print(" METHOD 3 : Since normalization in METHOD 1, add a phase to state, here, I transfprmed the")
print("            state into two-site-canonial form centered at Site(0,0) and Node(0,0) and divided both")
print("            by np.sqrt(<bra|ket>) and np.sqrt(<bra|ket>).conj()")

print(" ########## initialize the TDVP ###########")
tdvp_Lindblad.state = ptn.normalize_ttn_Lindblad_A(tdvp_Lindblad.state, tdvp_Lindblad.connections )
tdvp_Lindblad._orthogonalize_init()
tdvp_Lindblad.partial_tree_cache = tdvp_Lindblad._init_partial_tree_cache()

tdvp_Lindblad.adjust_to_initial_structure()

print(" <O>                           : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N))
print(" <bra|ket> , np.abs(<bra|ket>) : " , ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) , np.abs(ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) ) )
print(" <O> /<bra|ket>                : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N) / ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) )

for i in range(10):
    print(" ||||||||||||||||||| START STEP : " , i , " ||||||||||||||||||||||")

    print(" ######### RUN ONE STEP #########")
    tdvp_Lindblad.run_one_time_step()
    tdvp_Lindblad.adjust_to_initial_structure()
    print(" <O>                           : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N))
    print(" <bra|ket> , np.abs(<bra|ket>) : " , ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) , np.abs(ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) ) )
    print(" <O> /<bra|ket>                : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N) / ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) )

    print(" ######### NORMALIZATION ##########")
    tdvp_Lindblad.state = ptn.normalize_ttn_Lindblad_A(tdvp_Lindblad.state, tdvp_Lindblad.connections )
    print(" <O>                           : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N))
    print(" <bra|ket> , np.abs(<bra|ket>) : " , ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) , np.abs(ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) ) )
    print(" <O> /<bra|ket>                : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N) / ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) )

    print(" ######### orth. init and update treecache dict. with normalized state ##########") 
    tdvp_Lindblad._orthogonalize_init(force_new=True)
    tdvp_Lindblad.partial_tree_cache = ptn.PartialTreeCachDict()
    tdvp_Lindblad.partial_tree_cache = tdvp_Lindblad._init_partial_tree_cache()
    print(" <O>                           : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N))
    print(" <bra|ket> , np.abs(<bra|ket>) : " , ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) , np.abs(ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) ) )
    print(" <O> /<bra|ket>                : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N) / ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) )

    print(" ||||||||||||||||||| END STEP    : " , i , " |||||||||||||||||||||")




 METHOD 3 : Since normalization in METHOD 1, add a phase to state, here, I transfprmed the
            state into two-site-canonial form centered at Site(0,0) and Node(0,0) and divided both
            by np.sqrt(<bra|ket>) and np.sqrt(<bra|ket>).conj()
 ########## initialize the TDVP ###########
 <O>                           :  (5.7600000000000104+0j)
 <bra|ket> , np.abs(<bra|ket>) :  (1.0000000000000018+0j) 1.0000000000000018
 <O> /<bra|ket>                :  (5.76+0j)
 ||||||||||||||||||| START STEP :  0  ||||||||||||||||||||||
 ######### RUN ONE STEP #########
 <O>                           :  (5.331239741166343+0.0038076193886073056j)
 <bra|ket> , np.abs(<bra|ket>) :  (0.9246283439923833-8.44782933435486e-06j) 0.9246283440309747
 <O> /<bra|ket>                :  (5.765819034828414+0.004170679028868357j)
 ######### NORMALIZATION ##########
 <O>                           :  (5.765819034828412+0.004170679028868358j)
 <bra|ket> , np.abs(<bra|ket>) :  (0.9999999999999991+1.79570984817

In [20]:
print(" METHOD 4 : tdvp.state is not normalized at each step, instead the normalization factor per site")
print("            (<bra|ket> ** (1/2*number of sites)) is considered in expectation_value_Lindblad_2.")

print(" ########## initialize the TDVP ###########")
tdvp_Lindblad._orthogonalize_init()
tdvp_Lindblad.partial_tree_cache = tdvp_Lindblad._init_partial_tree_cache()
tdvp_Lindblad.adjust_to_initial_structure()
print(" <O> with local normalization  : " , ptn.expectation_value_Lindblad_2(tdvp_Lindblad.state , tdvp_Lindblad.connections, N))
print(" <O>                           : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N))
print(" <bra|ket> , np.abs(<bra|ket>) : " , ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) , np.abs(ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) ) )
print(" <O> /<bra|ket>                : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N) / ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) )

for i in range(10):
    print(" ||||||||||||||||||| START STEP : " , i , " ||||||||||||||||||||||")

    print(" ######### RUN ONE STEP #########")
    tdvp_Lindblad.run_one_time_step()
    tdvp_Lindblad.adjust_to_initial_structure()
    print(" <O> with local normalization  : " , ptn.expectation_value_Lindblad_2(tdvp_Lindblad.state , tdvp_Lindblad.connections, N))
    print(" <O>                           : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N))
    print(" <bra|ket> , np.abs(<bra|ket>) : " , ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) , np.abs(ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) ) )
    print(" <O> /<bra|ket>                : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N) / ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) )

    print(" ######### orth. init and update treecache dict. ##########") 
    tdvp_Lindblad._reset_for_next_time_step()
    tdvp_Lindblad.adjust_to_initial_structure()
    print(" <O> with local normalization  : " , ptn.expectation_value_Lindblad_2(tdvp_Lindblad.state , tdvp_Lindblad.connections, N))
    print(" <O>                           : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N))
    print(" <bra|ket> , np.abs(<bra|ket>) : " , ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) , np.abs(ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) ) )
    print(" <O> /<bra|ket>                : " , ptn.expectation_value_Lindblad_1(tdvp_Lindblad.state , tdvp_Lindblad.connections, N) / ptn.bra_ket(tdvp_Lindblad.state , tdvp_Lindblad.connections) )

    print(" ||||||||||||||||||| END STEP    : " , i , " |||||||||||||||||||||")




 METHOD 4 : tdvp.state is not normalized at each step, instead the normalization factor per site
            (<bra|ket> ** (1/2*number of sites)) is considered in expectation_value_Lindblad_2.
 ########## initialize the TDVP ###########
 <O> with local normalization  :  (6.372660764462142+0j)
 <O>                           :  (6.372660764462142+0j)
 <bra|ket> , np.abs(<bra|ket>) :  (1+0j) 1.0
 <O> /<bra|ket>                :  (6.372660764462142+0j)
 ||||||||||||||||||| START STEP :  0  ||||||||||||||||||||||
 ######### RUN ONE STEP #########
 <O> with local normalization  :  (6.385160385379106+0.002016017558191019j)
 <O>                           :  (6.715785354982947+0.002120407378284496j)
 <bra|ket> , np.abs(<bra|ket>) :  (1.0464837686167991+3.494369196671878e-05j) 1.0464837692002107
 <O> /<bra|ket>                :  (6.417476906665439+0.0018119316313531676j)
 ######### orth. init and update treecache dict. ##########
 <O> with local normalization  :  (6.385160385379107+0.00201601755

In [263]:
from qutip import *
import numpy as np

# Define parameters
t = 0.4  # Hopping strength
U = 0.8  # On-site interaction strength
m = 0.4  # Chemical potential
gamma_relax = 1  # Relaxation rate

# Reduced lattice dimensions
Nx = 2  # Number of sites along x-direction
Ny = 2  # Number of sites along y-direction
N = Nx * Ny  # Total number of sites

# Reduced maximum number of bosons per site
nmax = 1

# Precompute the operators for each site
a_list = []
adag_list = []
n_list = []
si = qeye(nmax + 1)  # Identity operator for a single site
for n in range(N):
    op_list = [si] * N
    op_list[n] = destroy(nmax + 1)
    a_op = tensor(op_list)
    a_list.append(a_op)
    adag_list.append(a_op.dag())
    n_list.append(a_op.dag() * a_op)


# Function to map 2D lattice coordinates (i, j) to a site index
def site(i, j):
    return i + j * Nx

# Initialize the Hamiltonian
H = 0

# Build the Hamiltonian by summing over sites
for i in range(Nx):
    for j in range(Ny):
        n = site(i, j)
        H += 0.5 * U * n_list[n] * (n_list[n] - 1) - m * n_list[n]
        if i < Nx - 1:
            n_right = site(i + 1, j)
            H += -t * (adag_list[n] * a_list[n_right] + adag_list[n_right] * a_list[n])
        if j < Ny - 1:
            n_up = site(i, j + 1)
            H += -t * (adag_list[n] * a_list[n_up] + adag_list[n_up] * a_list[n])

# Initial state: product state of maximum occupation

#psi0 = tensor([basis(nmax + 1, 1) for _ in range(N)]).unit()
psi0 = tensor([(3 *basis(nmax + 1, 0) + 4j* basis(nmax + 1, 1)).unit() for _ in range(N)])
alpha = 1
psi0 = 1j *tensor([coherent(nmax + 1, alpha) for _ in range(N)])
# Reduced simulation time and increased time step
total_time = 0.1  # Total time in seconds
time_step = 0.01  # Time step in seconds
tlist = np.arange(0, total_time + time_step, time_step)

# Define collapse operators (for the Lindblad equation)
custom_matrix = Qobj([[0, 1], [0, 0]])
jump_operator = []
si = qeye(nmax + 1) 
for n in range(N):
  op_list = [si] * N  # Create a list of identity operators
  op_list[n] = custom_matrix  # Replace the n-th site with the custom matrix
  custom_op = tensor(op_list)  # Create the tensor product
  jump_operator.append(custom_op)

c_ops = [np.sqrt(gamma_relax) * a for a in a_list]


# Observables to calculate - total particle number
N_total = sum(n_list)

# Solve the Schrödinger equation (more efficient for this case)
result = mesolve(H, psi0, tlist, c_ops, [N_total])

# Extract expectation values
total_number = result.expect[0]

# Print results
print("Time evolution of total particle number:")
for t, n in zip(tlist, total_number):
    print(f"Time: {t:.2f}, Total number: {n:.4f}")

Time evolution of total particle number:
Time: 0.00, Total number: 2.8323
Time: 0.01, Total number: 2.8041
Time: 0.02, Total number: 2.7762
Time: 0.03, Total number: 2.7486
Time: 0.04, Total number: 2.7212
Time: 0.05, Total number: 2.6942
Time: 0.06, Total number: 2.6674
Time: 0.07, Total number: 2.6408
Time: 0.08, Total number: 2.6145
Time: 0.09, Total number: 2.5885
Time: 0.10, Total number: 2.5628


In [ ]:
import matplotlib.pyplot as plt

fig1, axs1 = plt.subplots(1, 1, sharex=True, figsize=(5, 5))

axs1.plot( tdvp_Lindblad.operator_results()[0] , label="tdvp")
axs1.plot( np.imag(tdvp_Lindblad.operator_results()[0]) , label="tdvp_imag")

axs1.plot( total_number , label="Exact")

axs1.set_xlabel("Time $t$")
axs1.set_ylabel("Total Occupation ")
axs1.grid(True)
axs1.legend()